# 基礎編6

このノートブックでは、グループの管理と、アクセス権限での利用について説明します。

## 準備

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { loadWallet } = await import('../lib/load-wallet.mjs');
var { domain } = await import('../lib/load-config.mjs');
var { adminWallet, rpc, deploySmartContract } = await import('../lib/notebook-util.mjs');

あらかじめ用意してあるユーザ／ウォレットを読み込みます。

In [2]:
var users = [];
for(var i=0; i<=5; i++){
    var wfstr = await loadWallet(`wallet-user${i}.json`);
    var wallet = await api.unlockWalletFile(await api.parseWalletFile(wfstr), '_paswrd_');
    var id = (await rpc.call(wallet, 'c1query', { type: 'search', key: 'me' })).value[0].id;
    users[i] = { id, wallet };
    console.log(`user${i}:`, id, wallet.address);
}

user0: u73621973 eQiA3bsfZunQ8KsRpkTgMGndwu46rq
user1: u28169743 epGyEm4BXFxFvLmXkDdDvcqspESPFu
user2: u53371386 eaZMspgRpi68EdReTZEiUkdJuKsdct
user3: u58185320 e5fVm5R4eKYXa7U57pshS4UQwuvGsp
user4: u10676071 eEYRycGm4znXxnf7bTngX72JmYk57r
user5: u00571239 eWW6LCfRSpoVuudzZHa4ZwCjkkizov


## グループの作成とメンバの登録・削除

グループを新規に作成します。

In [3]:
var resp = await rpc.call(adminWallet, 'c1create', { type: 'group', domain });
console.log(resp);
var gid = resp.value;

{
  txno: 701701,
  txid: 'xLzL5B8N3FRW9CMzXGEpsLpPnmBonbRxoQTauXkPRusRNB',
  status: 'ok',
  value: 'g15229146'
}


作成直後はグループのメンバは空です。

In [4]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'group_members', id: gid });
console.log(resp);

{
  txid: 'xJFkdXE7dabAPwyBvamdUgQjbncUa7zUnC99b4keX55Nx',
  status: 'ok',
  value: { list: [] }
}


メンバとしてuser0を追加します。

In [5]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [ users[0].id ] });
console.log(resp);

{
  txno: 701702,
  txid: 'xtY76viv8fMt6xGcyB7WZxZWusyz2ZDvtjQesvJqba9P2',
  status: 'ok',
  value: null
}


グループのメンバに追加されたことを確認します。

In [6]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'group_members', id: gid });
console.log(resp);

{
  txid: 'xiGEfSpJwUjaRCh4DuxiChqqdDH567HdyZ4eEhnihr3v5',
  status: 'ok',
  value: {
    list: [
      {
        id: [ 'u73621973', '_test_user0@c.TDSL.H011' ],
        c_txno: 701607,
        created_at: 1753338670120
      }
    ]
  }
}


さらに、user1とuser2を追加します。

In [7]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [ users[1].id, users[2].id ] });
console.log(resp);

{
  txno: 701703,
  txid: 'xmiP9qdx2PUDB5fzjRgczoWTsj8SrDp2kK9ZBDw7HmFct',
  status: 'ok',
  value: null
}


追加されたことを確認します。

In [8]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'group_members', id: gid });
console.log(resp);

{
  txid: 'xWC9YKcpuw226JouosEaR6DgFXEAUTw3t3dHH5758cdEv',
  status: 'ok',
  value: {
    list: [
      {
        id: [ 'u53371386', '_test_user2@c.TDSL.H011' ],
        c_txno: 701611,
        created_at: 1753338675203
      },
      {
        id: [ 'u28169743', '_test_user1@c.TDSL.H011' ],
        c_txno: 701609,
        created_at: 1753338672648
      },
      {
        id: [ 'u73621973', '_test_user0@c.TDSL.H011' ],
        c_txno: 701607,
        created_at: 1753338670120
      }
    ]
  }
}


グループのメンバからuser1を削除します。

In [9]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'delete members', value: [ users[1].id ] });
console.log(resp);

{
  txno: 701704,
  txid: 'xK5dQfd9jM9AhtvZt3GNguDpsPfKjoLmZtWnLjzwZdppCB',
  status: 'ok',
  value: null
}


削除されたことを確認します。

In [10]:
var resp = await rpc.call(adminWallet, 'c1query', { type: 'group_members', id: gid });
console.log(resp);

{
  txid: 'xFyGGNdhdWuKt6maqRZQrHuCaWriwo3J9rYA6hhsYTtyv',
  status: 'ok',
  value: {
    list: [
      {
        id: [ 'u53371386', '_test_user2@c.TDSL.H011' ],
        c_txno: 701611,
        created_at: 1753338675203
      },
      {
        id: [ 'u73621973', '_test_user0@c.TDSL.H011' ],
        c_txno: 701607,
        created_at: 1753338670120
      }
    ]
  }
}


## グループを使った権限の設定

テスト用に簡単なスマートコントラクトをデプロイします。

In [11]:
var cid = await deploySmartContract(function basic6(){
    return "呼び出し成功";
});

user0が実行しようとするとエラーとなることを確認します。

In [12]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);

{
  txno: 701708,
  txid: 'xAkzjgiVkTxPKGkY6pHVBicFoZKFyVbYWv8Q29kiXapgq',
  status: 'denied',
  value: 'accessible permission'
}


スマートコントラクトの実行権(accessible_to)に、上で作成したグループgidを追加します。

In [13]:
var resp = await rpc.call(adminWallet, 'c1update', { id: cid, prop: 'add accessible_to', value: [ gid ] });
console.log(resp);

{
  txno: 701709,
  txid: 'xbdor9pu7frAXZgWghNP7gUn9bbEEp2higmwiGwEkrPy4',
  status: 'ok',
  value: null
}


user0がグループgidに含まれており、グループgidに実行権が与えられたため、user0による実行が今度は成功します。

In [14]:
var resp = await rpc.call(users[0].wallet, cid);
console.log(resp);

{
  txno: 701710,
  txid: 'xQ32Jho5WfJZCAuMvtog3ZDF9iw4GBtfsfdRAU4ZyQzuKB',
  status: 'ok',
  value: '呼び出し成功'
}


次に、グループにメンバをあとから追加するパターンを示します。  
user3が実行しようとするとエラーとなることを確認します。

In [15]:
var resp = await rpc.call(users[3].wallet, cid);
console.log(resp);

{
  txno: 701711,
  txid: 'xQD6hyZ953gLf3seUtZnHGjJFt3aUDZegt7hFsA59AhZr',
  status: 'denied',
  value: 'accessible permission'
}


user3をグループgidにメンバとして加えます。

In [16]:
var resp = await rpc.call(adminWallet, 'c1update', { id: gid, prop: 'add members', value: [ users[3].id ] });
console.log(resp);

{
  txno: 701712,
  txid: 'xYNRE8R3tPxB9jzoM4q7gYTnWrYFaePhHVtFEdTWWVeU5',
  status: 'ok',
  value: null
}


user3による実行が今度は成功します。  
このように、あとからグループに追加した場合でも権限が付与されます。

In [17]:
var resp = await rpc.call(users[3].wallet, cid);
console.log(resp);

{
  txno: 701713,
  txid: 'xijcUhAF8iAYskKj7p88esvifHoRq29NkBW4EpG3hGkSp',
  status: 'ok',
  value: '呼び出し成功'
}


## クリーンアップ
このノートブックの中で作成したグループは今後使用しないので、削除しておきます。

In [18]:
var resp = await rpc.call(adminWallet, 'c1terminate', { id: gid });
console.log(resp);

{
  txno: 701714,
  txid: 'xP84LuihUCQD2mSuFYQrye4a2UMoqpZ8TgyJ4sbb8t7DEB',
  status: 'ok',
  value: 'g15229146'
}
